In [39]:
import pandas as pd
import geopandas as gpd
import numpy as np

pd.set_option("display.float_format", "{:.4f}".format)

In [40]:
ABT = gpd.read_file(
    "../../../../Data/Final_dataset/ABT/ABT.gpkg",
    layer="subdivisions"
)

npa_raw = gpd.read_file(
    "../../../../Data/Original_dataset/original.gdb",
    layer="QOL_NPA_2020_final_projected"
)

In [41]:
ABT_proj = ABT[ABT["year"].between(1990, 2023)].copy()
npa_proj = npa_raw.to_crs(ABT_proj.crs)

# Areas (in CRS units; ideally a projected CRS in meters)
ABT_proj["subd_area"] = ABT_proj.geometry.area
npa_proj["npa_area"] = npa_proj.geometry.area

In [43]:
abt_npa_intersections = gpd.overlay(
    ABT_proj[["subd_id", "subd_area", "geometry"]],
    npa_proj[["NPA_ID", "geometry"]],
    how="intersection"
)

abt_npa_intersections["intersect_area"] = abt_npa_intersections.geometry.area
abt_npa_intersections["share_intersect"] = (
    100.0 * abt_npa_intersections["intersect_area"] / abt_npa_intersections["subd_area"]
)

abt_npa_intersections

,subd_id,subd_area,NPA_ID,geometry,intersect_area,share_intersect
0,4294,1852661.3399,299.0000,"POLYGON ((1489392.247 570674.813, 1489333.122 ...",1852459.4060,99.9891
1,4294,1852661.3399,377.0000,"MULTIPOLYGON (((1489518.226 571940.539, 148949...",201.9338,0.0109
2,2,570947.5917,273.0000,"POLYGON ((1496959.122 569527.438, 1496981.247 ...",570947.5917,100.0000
3,3,1197621.6359,301.0000,"POLYGON ((1491443.478 567036.821, 1491439.45 5...",8692.8458,0.7258
4,3,1197621.6359,377.0000,"POLYGON ((1492293.35 568254.921, 1492295.336 5...",1188928.7902,99.2742
...,...,...,...,...,...,...
7322,10179,59171.2458,396.0000,"POLYGON ((1530431.596 529956.42, 1530485.637 5...",59171.2458,100.0000
7323,10180,56543.5424,396.0000,"POLYGON ((1530023.061 529408.386, 1530082.57 5...",56543.5424,100.0000
7324,10181,51362.6493,396.0000,"POLYGON ((1530089.373 529224.496, 1530113.347 ...",51362.6493,100.0000
7325,10182,39914.9903,396.0000,"POLYGON ((1530089.373 529224.496, 1530132.87 5...",39914.9903,100.0000


In [44]:
df = abt_npa_intersections.copy()

# Sort so the largest intersection appears first
df = df.sort_values(["subd_id", "share_intersect"], ascending=[True, False])

# Rank intersections within each subdivision
df["npa_rank"] = df.groupby("subd_id").cumcount() + 1

In [45]:
df = df[df["npa_rank"] <= 6]

In [46]:
# Pivot area
area_wide = df.pivot(
    index="subd_id",
    columns="npa_rank",
    values="intersect_area"
)

# Pivot share
share_wide = df.pivot(
    index="subd_id",
    columns="npa_rank",
    values="share_intersect"
)

In [47]:
area_wide.columns = [f"npa{i}_area" for i in area_wide.columns]
share_wide.columns = [f"npa{i}_share" for i in share_wide.columns]

In [48]:
wide = pd.concat([area_wide, share_wide], axis=1)

In [49]:
# Total intersection per subdivision
total_intersection = (
    abt_npa_intersections
    .groupby("subd_id")["intersect_area"]
    .sum()
)

# Merge subdivision area
subd_area = (
    abt_npa_intersections
    .drop_duplicates("subd_id")
    .set_index("subd_id")["subd_area"]
)

outside_area = subd_area - total_intersection
outside_share = 100 * outside_area / subd_area

wide["intersect_area_outside"] = outside_area
wide["share_intersect_outside"] = outside_share

In [50]:
final_table = (
    abt_npa_intersections
    .drop_duplicates("subd_id")[["subd_id"]]
    .merge(wide, left_on="subd_id", right_index=True, how="left")
)

In [53]:
# Select only the 14 new columns
cols = [c for c in wide.columns]

summary_stats = (
    wide[cols]
    .describe()
    .T
)

summary_stats

,count,mean,std,min,25%,50%,75%,max
npa1_area,5844.0000,730232.6963,1975386.0218,3421.0810,53392.0798,197694.3162,696764.4515,66207107.8361
npa2_area,1198.0000,99693.4516,635755.3542,0.0008,33.2574,168.8217,3994.7442,11020865.9525
npa3_area,240.0000,21372.8699,145088.2617,0.0007,10.9016,50.3051,231.4253,1513208.3549
npa4_area,37.0000,4389.1989,19539.8753,0.0054,1.5176,70.2895,199.1393,116741.2462
npa5_area,7.0000,194.3622,401.9296,0.0130,1.6028,33.3352,113.6911,1096.5995
npa6_area,1.0000,17.8329,NaN,17.8329,17.8329,17.8329,17.8329,17.8329
npa1_share,5844.0000,99.2788,4.7865,15.1833,100.0000,100.0000,100.0000,100.0000
npa2_share,1198.0000,3.0125,8.7408,0.0000,0.0041,0.0218,0.4358,49.9951
npa3_share,240.0000,0.4061,2.2661,0.0000,0.0007,0.0031,0.0146,23.0708
npa4_share,37.0000,0.1728,0.9079,0.0000,0.0000,0.0019,0.0055,5.5233


In [54]:
npa_counts = (
    abt_npa_intersections
    .groupby("subd_id")["NPA_ID"]
    .nunique()
)

# Subdivisions fully inside one NPA
one_npa = (npa_counts == 1).sum()

# Total subdivisions
total_subd = npa_counts.shape[0]

percentage = 100 * one_npa / total_subd

one_npa, total_subd, percentage

(4646, 5844, 79.5003422313484)

In [56]:
# Example placeholder characteristics
npa_chars = npa_raw[["NPA_ID"]].copy()

npa_chars["char_income"] = np.random.uniform(40000, 90000, len(npa_chars))
npa_chars["char_density"] = np.random.uniform(1, 20, len(npa_chars))
npa_chars["char_transit"] = np.random.uniform(0, 1, len(npa_chars))

In [57]:
df = abt_npa_intersections.merge(
    npa_chars,
    on="NPA_ID",
    how="left"
)

In [58]:
df["area_weight"] = df["intersect_area"] / df["subd_area"]

In [59]:
char_cols = ["char_income", "char_density", "char_transit"]

for col in char_cols:
    df[f"{col}_weighted"] = df[col] * df["area_weight"]

In [60]:
sub_attribution = (
    df.groupby("subd_id")[
        [f"{c}_weighted" for c in char_cols]
    ]
    .sum()
    .reset_index()
)

# Rename cleanly
sub_attribution.columns = ["subd_id"] + char_cols

In [61]:
sub_attribution

,subd_id,char_income,char_density,char_transit
0,2,64253.3026,7.6765,0.6682
1,3,60277.5525,4.9323,0.2773
2,4,77336.1849,4.9188,0.3540
3,8,77336.1849,4.9188,0.3540
4,15,64253.3026,7.6765,0.6682
...,...,...,...,...
5839,10179,85123.2455,16.0800,0.7981
5840,10180,85123.2455,16.0800,0.7981
5841,10181,85123.2455,16.0800,0.7981
5842,10182,85123.2455,16.0800,0.7981
